# DESAFIO IDICIUM.AI


----
# Questão 1 - EDA

## Parte 1 - Visão geral da tabela orders

• Quantidade total de linhas

• Quantidade total de colunas

• Intervalo de datas analisado (data mínima e máxima) da coluna created_at

In [10]:
import pandas as pd
from pathlib import Path

csv_file = r'1-lh_nautical_csv\orders.csv'
df = pd.read_csv(csv_file)
df["created_at"] = pd.to_datetime(df["created_at"])

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
df["created_at_derivada"] = pd.to_datetime(df['created_at'],errors='coerce') #transforma datas inválidas em NaT (Not a Time)
print(f"Datas inválidas: {df['created_at_derivada'].isna().sum()}") #somando as datas invalidas
print(f"Período: {df['created_at'].min()} até {df['created_at'].max()}\n")

print(f"Nome e tipo colunas: {df.dtypes.to_dict()}")


Linhas: 48998
Colunas: 13
Datas inválidas: 0
Período: 2020-01-01 01:19:28 até 2026-12-31 23:43:09

Nome e tipo colunas: {'id': dtype('int64'), 'order_number': dtype('O'), 'channel': dtype('O'), 'customer_id': dtype('int64'), 'salesperson_id': dtype('float64'), 'location_id': dtype('int64'), 'status': dtype('O'), 'subtotal': dtype('float64'), 'discount_amount': dtype('float64'), 'total': dtype('float64'), 'placed_at': dtype('O'), 'created_at': dtype('<M8[ns]'), 'updated_at': dtype('O'), 'created_at_derivada': dtype('<M8[ns]')}


----


## Parte 2 - Análise de valores numéricos

Para a coluna "total", calcule:

• Valor mínimo

• Valor máximo

• Valor médio33 

In [11]:
print("\n=== COLUNA TOTAL (R$) ===")
print(f"Valor mínimo: {df['total'].min()}")
print(f"Valor máximo: {df['total'].max()}")
print(f"Valor médio: {df['total'].mean():.2f}")
print(f"Valor mediano: {df['total'].median()}")
print(f"Valores nulos: {df['total'].isna().sum()}")
zeros = (df['total'] == 0).sum()
print(f"Valores iguais a zero: {zeros}")

# Outliers
q1 = df['total'].quantile(0.25)
q3 = df['total'].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = df['total'][
    (df['total'] < limite_inferior) |
    (df['total'] > limite_superior)
]

print("\n=== OUTLIERS ===")
print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")
print(f"Possíveis outliers: {len(outliers)}")
print(f"Percentual: {len(outliers) / len(df['total']) * 100:.2f}%")


=== COLUNA TOTAL (R$) ===
Valor mínimo: 32.62
Valor máximo: 127262.02
Valor médio: 28704.99
Valor mediano: 25917.84
Valores nulos: 0
Valores iguais a zero: 0

=== OUTLIERS ===
Limite inferior: -28484.74
Limite superior: 82597.85
Possíveis outliers: 452
Percentual: 0.92%


## Parte 3 - Interpretação

Responda de forma resumida: *Com base na análise exploratória realizada, escreva um breve diagnóstico sobre a confiabilidade da tabela o para análises futuras.*

Comente sobre:
    
• possíveis outliers em "total"?

Sim. 452 registros (0,92%), entre 82.618,00 e 127.262,02 podem ser outliers. Mas esses valores precisam ser investigados, antes de decidir por exclusão ou tratamento. Possivelmente são valores válidos. OBS: não há valores menores que zero na coluna.

• qualidade dos dados (valores nulos ou inconsistentes)?

Boa, pois zero valores nulos e zero valores = 0 na coluna tota. Mas requer investigação no valores extremos apontados como outliers. Não há nulos em total e sem datas inválidas em _created_at_

• e se você considera que o dataset está pronto para análises ou se exigiria tratamento prévio?

Está adequado para uma EDA inicial, mas recomenda-se validação e tratamento prévio dos valores acima do limite superior, antes de modelagem.

In [12]:
outliers = df[df["total"] > limite_superior]

print(outliers[[
    "id",
    "order_number",
    "subtotal",
    "discount_amount",
    "total",
    "created_at"
]].sort_values("total", ascending=False).head(20))

          id order_number   subtotal  discount_amount      total  \
24584  25100    SO-025100  127262.02             0.00  127262.02   
24299  24810    SO-024810  125273.92             0.00  125273.92   
22858  23351    SO-023351  118037.91             0.00  118037.91   
30628  31266    SO-031266  118676.15          4747.05  113929.10   
34339  35048    SO-035048  113622.80             0.00  113622.80   
39206  40016    SO-040016  113317.69             0.00  113317.69   
13350  13625    SO-013625  113235.65             0.00  113235.65   
41265  42121    SO-042121  112683.09             0.00  112683.09   
45597  46535    SO-046535  112174.03             0.00  112174.03   
46291  47241    SO-047241  111899.74             0.00  111899.74   
26890  27454    SO-027454  111160.95             0.00  111160.95   
16572  16923    SO-016923  109612.25             0.00  109612.25   
8486    8665    SO-008665  118840.58          9507.25  109333.33   
47357  48327    SO-048327  109273.14            

## Questão 1.1 - SQL
Código calculando:

•	Quantidade total de linhas

•	Intervalo de datas analisado (data mínima e máxima)

•	Valor mínimo

•	Valor máximo

•	Valor médio

In [28]:
import sqlite3
import pandas as pd

# Carrega o CSV e armazena em memória
df = pd.read_csv(r'1-lh_nautical_csv\orders.csv')
conexao = sqlite3.connect(':memory:')
df.to_sql('orders', conexao, index=False, if_exists='replace')

query = """
SELECT 
    COUNT(*) AS total_linhas,
    MIN(created_at) AS data_minima,
    MAX(created_at) AS data_maxima,
    MIN(total) AS valor_minimo,
    MAX(total) AS valor_maximo,
    AVG(total) AS valor_medio
FROM orders;
"""

resultado = pd.read_sql(query, conexao)
print(resultado)
conexao.close()

   total_linhas          data_minima          data_maxima  valor_minimo  \
0         48998  2020-01-01 01:19:28  2026-12-31 23:43:09         32.62   

   valor_maximo   valor_medio  
0     127262.02  28704.992077  


---
# Questão 2 - Schema

Regra a seguir:

buscar CSV -> ler estrutura -> identificar colunas -> inferir tipos -> CREATE TABLE

**OBS:** código alterado quatro vezes, pois estava errando na identificação do tipo. Confundindo valor observado e regra semântica. Campos com numeros muito longos, como CPF, estavam sendo identificados como INTEGER no schema. O melhor seria TEXT.

Nova regra:

1. Detectar pelo CONTEÚDO
2. Regra semântica valida a escolha
3. Manter TEXT como forma mais segura

In [31]:
import os
import csv
from datetime import datetime

# ---------------------------------------------------------
# DETECÇÃO DOS TIPOS DE DADOS
# ---------------------------------------------------------

def eh_inteiro(valor):
    """Verifica se o valor pode ser convertido para inteiro."""
    if valor.strip() == "":
        return False

    try:
        int(valor)
        return True
    except ValueError:
        return False

def eh_decimal(valor):
    """Verifica se o valor pode ser convertido para decimal."""
    if valor.strip() == "":
        return False

    try:
        float(valor)
        return True
    except ValueError:
        return False

def eh_data(valor):
    """Verifica se o valor possui um formato de data conhecido."""
    if valor.strip() == "":
        return False

    formatos = [
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%Y/%m/%d"
    ]

    for formato in formatos:
        try:
            datetime.strptime(valor, formato)
            return True
        except ValueError:
            pass

    return False

# ------
# Nova regra adicionada para validar o conteúdo
def regra_semantica(nome_coluna):
    """Identifica o tipo pelo significado do nome da coluna."""

    nome = nome_coluna.lower().strip()

    # Identificadores que devem ser TEXT
    if nome in [
        "cpf",
        "cnpj",
        "tax_id",
        "phone",
        "telefone",
        "barcode",
        "barcode_ean",
        "ean",
        "access_key",
        "nfe_access_key",
        "chave_acesso",
        "ncm_code"
    ]:
        return "TEXT"

    # Campos de quantidade devem ser INTEGER
    # OBS: Mudança no código, pois nas tabelas aparecem valores
    #      com casas decimais (ex: 1.000), então é melhor usar NUMERIC
    if nome in [
        "quantidade",
        "qty",
        "quantity",
        "reorder_point", 
        "quantity_received", 
        "quantity_ordered"
    ]:
        return "NUMERIC(12,2)"

    # Campos de valor devem ser NUMERIC
    if (
        nome.endswith("_price")
        or nome.endswith("_amount")
        or nome.endswith("_value")
        or nome.endswith("_rate")
        or nome in ["price", "amount", "rate", "preco", "valor"]
    ):
        return "NUMERIC(12,2)"

    # Campos de data
    if (
        nome.startswith("date_")
        or nome.endswith("_date")
        or nome.endswith("_data") 
    ):
        return "DATE"

    # Campos de identificação (IDs) são INTEGER
    if nome == "id" or nome.endswith("_id"):
        return "INTEGER"

    # Campos booleanos
    if nome.startswith("is_"):
        return "BOOLEAN"
    
    return None


def detectar_tipo(nome_coluna, valores):
    """Define o tipo SQL com base nos seus valores ou regra semântica (coluna)."""

    # PASSO 1: pelo conteúdo (valores)
    # ---------------------
    valores_validos = [valor for valor in valores if valor.strip()]

    if not valores_validos:
        tipo_conteudo = "TEXT"

    elif all(eh_inteiro(valor) for valor in valores_validos):
        tipo_conteudo = "INTEGER"

    elif all(eh_decimal(valor) for valor in valores_validos):
        tipo_conteudo = "NUMERIC(12,2)"

    elif all(eh_data(valor) for valor in valores_validos):
        tipo_conteudo = "DATE"

    else: # Se não identificar um tipo específico, TEXT como opção mais segura.
        tipo_conteudo = "TEXT"


    # PASSO 2: validar pela semântica
    # ------------------------------
    tipo_semantico = regra_semantica(nome_coluna)

    if tipo_semantico:
        return tipo_semantico # caso ache nos tipos

    return tipo_conteudo # senão fica o que foi identificado

# ---------------------------------------------------------
# LEITURA DOS CSVs
# ---------------------------------------------------------

def ler_csv(caminho):
    """Lê um CSV e retorna suas colunas e tipos detectados."""

    dados_colunas = {}

    print(f"Lendo: {caminho}")

    try:
        with open(caminho, "r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)  # DictReader lê todo o CSV como texto

            nomes_colunas = leitor.fieldnames

            if not nomes_colunas:
                print("  Arquivo sem cabeçalho.")
                return [], []

            # Criação de uma lista para guardar os valores de cada coluna.
            for coluna in nomes_colunas:
                dados_colunas[coluna] = []

            for linha in leitor:
                for coluna in nomes_colunas:
                    dados_colunas[coluna].append(linha[coluna])

    except Exception as erro:
        print(f"  Erro ao ler arquivo: {erro}")
        return [], []

    tipos = []

    for coluna in nomes_colunas:
        tipo = detectar_tipo(coluna, dados_colunas[coluna])
        tipos.append(tipo)
        print(f"  {coluna}: {tipo}")

    return nomes_colunas, tipos


# ---------------------------------------------------------
# GERAÇÃO DO SQL
# ---------------------------------------------------------

def nome_tabela(nome_arquivo):
    """Transforma o nome do CSV em nome de tabela."""

    nome = os.path.splitext(nome_arquivo)[0]

    nome = nome.lower()
    nome = nome.replace("-", "_")
    nome = nome.replace(" ", "_")

    return nome


def gerar_create_table(nome_arquivo, colunas, tipos):
    """Monta o CREATE TABLE correspondente ao CSV."""

    tabela = nome_tabela(nome_arquivo)

    sql = f"-- Tabela: {tabela}\n"
    sql += f"-- Arquivo: {nome_arquivo}\n"
    sql += f"CREATE TABLE {tabela} (\n"

    for i in range(len(colunas)):
        sql += f"    {colunas[i]} {tipos[i]}"

        if i < len(colunas) - 1:
            sql += ","

        sql += "\n"

    # Se existir uma coluna chamada id, considerando ela como chave primária.
    if "id" in colunas:
        sql += ",    PRIMARY KEY (id)\n"

    sql += ");\n"

    return sql


# ---------------------------------------------------------
# FUNÇÃO PRINCIPAL
# ---------------------------------------------------------

def gerar_schema(diretorio_csv, arquivo_saida):
    """Percorre os CSVs e gera o arquivo schema.sql."""

    if not os.path.isdir(diretorio_csv):
        print(f"Diretório não encontrado: {diretorio_csv}")
        return

    arquivos_csv = [
        arquivo
        for arquivo in os.listdir(diretorio_csv)
        if arquivo.lower().endswith(".csv")
    ]

    if not arquivos_csv:
        print("Nenhum arquivo CSV encontrado.")
        return

    sql_completo = ""

    for arquivo in sorted(arquivos_csv):

        caminho = os.path.join(diretorio_csv, arquivo)

        colunas, tipos = ler_csv(caminho)

        if not colunas:
            continue

        sql_completo += gerar_create_table(
            arquivo,
            colunas,
            tipos
        )

        sql_completo += "\n"

    # Gerando arquivo final com todas as tabelas encontradas.
    with open(arquivo_saida, "w", encoding="utf-8") as arquivo_sql:

        arquivo_sql.write("-- =========================================\n")
        arquivo_sql.write("-- SCHEMA SQL - LH NAUTICAL\n")
        arquivo_sql.write("-- Gerado automaticamente a partir dos CSVs\n")
        arquivo_sql.write(
            f"-- Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        )
        arquivo_sql.write("-- =========================================\n\n")

        arquivo_sql.write(sql_completo)

    print(f"\nSchema criado com sucesso: {arquivo_saida}")


# ---------------------------------------------------------
# EXECUÇÃO
# ---------------------------------------------------------

if __name__ == "__main__":

    DIRETORIO_CSV = "1-lh_nautical_csv"
    ARQUIVO_SAIDA = "schema_nautical.sql"

    gerar_schema(
        DIRETORIO_CSV,
        ARQUIVO_SAIDA
    )


Lendo: 1-lh_nautical_csv\addresses.csv
  id: INTEGER
  customer_id: INTEGER
  address_type: TEXT
  postal_code: TEXT
  street: TEXT
  number: INTEGER
  complement: TEXT
  district: TEXT
  city: TEXT
  state: TEXT
  country: TEXT
  is_primary: BOOLEAN
Lendo: 1-lh_nautical_csv\attributes.csv
  id: INTEGER
  name: TEXT
  data_type: TEXT
Lendo: 1-lh_nautical_csv\brands.csv
  id: INTEGER
  name: TEXT
  country: TEXT
  is_active: BOOLEAN
  created_at: DATE
  updated_at: DATE
Lendo: 1-lh_nautical_csv\categories.csv
  id: INTEGER
  name: TEXT
  slug: TEXT
  parent_category_id: INTEGER
  is_active: BOOLEAN
  created_at: DATE
  updated_at: DATE
Lendo: 1-lh_nautical_csv\customers.csv
  id: INTEGER
  person_type: TEXT
  legal_name: TEXT
  trade_name: TEXT
  tax_id: TEXT
  state_registration: TEXT
  email: TEXT
  phone: TEXT
  is_active: BOOLEAN
  created_at: DATE
  updated_at: DATE
Lendo: 1-lh_nautical_csv\employees.csv
  id: INTEGER
  full_name: TEXT
  cpf: TEXT
  email: TEXT
  role: TEXT
  prima

----
# Questão 3 - Carregamento

Carregar os aquivos .csv no SGBD PostgreSQL

PostgreSQL Version 18.6 para Windows x86-64 instalado

*pip install psycopg2-binary* (conexão com banco PostgreSQL)



In [32]:
import os
import psycopg2

# ---------------------------------------------------------
# CONFIGURAÇÕES
# ---------------------------------------------------------

HOST = "localhost"
PORTA = "5432"
BANCO = "nautical"
USUARIO = "postgres"
SENHA = "postgres"
DIRETORIO_CSV = "1-lh_nautical_csv"

# ---------------------------------------------------------
# CONEXÃO
# ---------------------------------------------------------

def conectar_banco():
    """Cria a conexão com o PostgreSQL."""

    return psycopg2.connect(
        host=HOST,
        port=PORTA,
        database=BANCO,
        user=USUARIO,
        password=SENHA
    )

# ---------------------------------------------------------
# NOME DA TABELA
# ---------------------------------------------------------

def nome_tabela(nome_arquivo):
    """Transforma o nome do CSV em nome de tabela."""

    nome = os.path.splitext(nome_arquivo)[0]

    nome = nome.lower()
    nome = nome.replace("-", "_")
    nome = nome.replace(" ", "_")

    return nome

# ---------------------------------------------------------
# CARREGAMENTO
# ---------------------------------------------------------

def carregar_csv(conexao, caminho, tabela):
    """Carrega um CSV na tabela com o mesmo nome copiando os tipos."""

    cursor = conexao.cursor()

    try:
        with open(caminho, "r", encoding="utf-8") as arquivo:

            comando = f"""
                COPY {tabela}
                FROM STDIN
                WITH CSV HEADER
            """

            cursor.copy_expert(comando, arquivo)

        conexao.commit()

        print(f"{tabela}: carregada com sucesso.")

    except Exception as erro:
        conexao.rollback()
        print(f"{tabela}: erro - {erro}")

    finally:
        cursor.close()

# ---------------------------------------------------------
# FUNÇÃO PRINCIPAL DE CARREGAMENTO
# ---------------------------------------------------------

def carregar_todos_csv():

    if not os.path.isdir(DIRETORIO_CSV):
        print(f"Diretório não encontrado: {DIRETORIO_CSV}")
        return

    arquivos_csv = [
        arquivo
        for arquivo in os.listdir(DIRETORIO_CSV)
        if arquivo.lower().endswith(".csv")
    ]

    if not arquivos_csv:
        print("Nenhum CSV encontrado.")
        return

    try:
        conexao = conectar_banco()

        print(f"{len(arquivos_csv)} arquivos encontrados.\n")

        for arquivo in sorted(arquivos_csv):

            caminho = os.path.join(
                DIRETORIO_CSV,
                arquivo
            )

            tabela = nome_tabela(arquivo)

            carregar_csv(
                conexao,
                caminho,
                tabela
            )

        conexao.close()

        print("\nCarregamento finalizado.")

    except Exception as erro:
        print(f"Erro ao conectar ao banco: {erro}")


# ---------------------------------------------------------
# EXECUÇÃO
# ---------------------------------------------------------

if __name__ == "__main__":
    carregar_todos_csv()

24 arquivos encontrados.

addresses: carregada com sucesso.
attributes: carregada com sucesso.
brands: carregada com sucesso.
categories: carregada com sucesso.
customers: carregada com sucesso.
employees: carregada com sucesso.
fiscal_invoices: carregada com sucesso.
goods_receipt_items: carregada com sucesso.
goods_receipts: carregada com sucesso.
locations: carregada com sucesso.
order_items: carregada com sucesso.
orders: carregada com sucesso.
payments: carregada com sucesso.
product_suppliers: carregada com sucesso.
product_variants: carregada com sucesso.
products: carregada com sucesso.
purchase_order_items: carregada com sucesso.
purchase_orders: carregada com sucesso.
return_items: carregada com sucesso.
returns: carregada com sucesso.
stock_levels: carregada com sucesso.
stock_movements: carregada com sucesso.
suppliers: carregada com sucesso.
variant_attribute_values: carregada com sucesso.

Carregamento finalizado.


----
**Validação**

Qual o total de linhas soma das das seguintes tabelas: customers, orders, order_items e payments?

In [33]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    database="nautical",
    user="postgres",
    password="postgres"
)

cursor = conexao.cursor()

print("Conexão realizada com sucesso!")

cursor.execute("""
    SELECT
        (SELECT COUNT(*) FROM customers) +
        (SELECT COUNT(*) FROM orders) +
        (SELECT COUNT(*) FROM order_items) +
        (SELECT COUNT(*) FROM payments)
        AS total_linhas;
""")

resultado = cursor.fetchone()

print(f"Total de linhas: {resultado[0]}")

Conexão realizada com sucesso!
Total de linhas: 251864


----
# Questão 4 - Análise de clientes

Premissas obrigatórias:

•	Faturamento Total: Soma da coluna total por cliente.

•	Frequência: Contagem total de transações (IDs de venda) por cliente.

•	Ticket Médio: Faturamento Total / Frequência.

•	Diversidade de Categorias: Quantidade de categorias distintas (category_id) que o cliente comprou.

•	Filtro de Elite: Apenas clientes que compraram produtos de 13 ou mais categorias distintas devem ser considerados no ranking.

•	Desempate: Em caso de empate no Ticket Médio, utilize o customer_id em ordem crescente.


## Questão 4.1 - Código SQL

Código calculando:

1. O Ticket Médio e a Diversidade de categorias por cliente.

2. A identificação e filtro dos 10 clientes "Fiéis" (maior Ticket Médio entre aqueles com diversidade >= 13 categorias).

3. Para este grupo específico de 10 clientes, identifique qual categoria de produto concentra a maior quantidade total de itens comprados (sum(quantity)).

**OBS: código SQL rodado no PostgresSQL**

**RESULTADO:**

| customer_id | faturamento_total | frequencia | ticket_medio | diversidade_categorias |
| --- | --- | --- | --- | --- |
| 22 | 1087838.44 | 26 | 41839.94 | 14 |
| 1477 | 916262.58 | 22 | 41648.30 | 14 |
| 929 | 1082775.89 | 26 | 41645.23 | 14 |
| 1116 | 655737.20 | 16 | 40983.58 | 14 |
| 1691 | 815471.30 | 20 | 40773.57 | 14 |
| 774 | 726127.99 | 18 | 40340.44 | 14 |
| 1470 | 1040553.09 | 26 | 40021.27 | 14 |
| 1599 | 997616.46 | 25 | 39904.66 | 14 |
| 965 | 677297.78 | 17 | 39841.05 | 14 |
| 1722 | 1146455.22 | 29 | 39532.94 | 14 |

**RESULTADO:**

| category_id |	categoria |	quantidade_total_itens |
| --- | --- | --- |
| 8 | Hélices | 492.00 |


## Questão 4.2 - Explique:

**1.	Como você chegou nas categorias mais vendidas? (mapeamento da cadeia de chaves)**

Mapeando a cadeia desde a tabela _orders_ até _categories_, utilizando as respectivas chaves estrangeiras (JOINS):

orders.id →  order_items.order_id → order_items.product_variant_id → 
product_variants.id → product_variants.product_id → 
products.id → products.category_id → categories.id

**2.	Qual lógica utilizou para filtrar os clientes com diversidade mínima?**

Foi criada a CTE diversidade, que calcula, para cada cliente, a quantidade de categorias distintas por meio de COUNT(DISTINCT p.category_id). A CTE clientes_elite realiza um INNER JOIN e aplica o filtro WHERE d.diversidade_categorias >= 13.

**3.	Como garantiu que a contagem de itens refletisse apenas os Top 10?**

A CTE clientes_elite ordena os clientes pelo ticket_medio e **aplica LIMIT 10**. O INNER JOIN com orders pelo customer_id garante que apenas as compras (orders) desses clientes_elite sejam consideradas na SUM(order_item.quantity).

----
# Questão 5 - Dimensão de calendário

## Questão 5.1 - Código SQL

•	Desenvolvimento de um calendário com os dias da semana (em portugues)

•	LEFT JOIN entre o calendário e a tabela de vendas agregação de vendas por dia (soma de valor_venda), substituição de valores nulos por zero para dias sem vendas


hierarquia: lojas físicas --> todos os dias

**OBS: código SQL rodado no PostgresSQL**

**RESULTADO:**

| dia_semana | media_valor_vendas |
| -- | -- |
| Quinta-feira | 157154.32 |
| Domingo |	157616.13 |
| Segunda-feira |	158241.15 |
| Sábado |	164858.27 |
| Terça-feira |	166118.83 |
| Sexta-feira |	170193.68 |
| Quarta-feira |	173605.44 |

# Questão 5.2 - Explique:

**1.	Por que é necessário utilizar uma tabela de datas (calendário) em vez de agrupar diretamente a tabela de vendas?**

Para garantir que todos os dias do período sejam considerados na análise. Se analisar somente tabela de vendas, apenas os dias lançados nas transações serão considerados.

**2.	O que aconteceria com a média de vendas se um dia da semana tivesse muitos dias sem nenhuma venda registrada?**

A média de vendas reduzirá, pois a média é sensível aos extremos, e como os dias sem vendas passam a ser considerados com valor zero no cálculo, haverá redução de valor médio.


----
# Questão 6 - Previsão de demanda

Premissas obrigatórias:

•	O período de treino deve incluir dados até 31/12/2025.

•	O período de teste deve ser o primeiro trimestre de 2026.

•	A previsão deve ser feita em base mensal.

•	Considere apenas o produto: "Bússola de Bordo 702"


In [75]:
'''
1. PostgreeSQL
   Datasets products.csv, product_variants, order_items.csv, orders.csv para criar um dataset unificado.
'''
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    database="nautical",
    user="postgres",
    password="postgres"
)

cursor = conexao.cursor()

print("Conexão realizada com sucesso!")

codigo_sql = """
    WITH vendas_produto AS (
        SELECT
            o.placed_at::date AS data_venda,
            oi.quantity
        FROM products p
        JOIN product_variants pv
            ON p.id = pv.product_id
        JOIN order_items oi
            ON pv.id = oi.product_variant_id
        JOIN orders o
            ON o.id = oi.order_id
        WHERE p.id in (74, 240) -- códigos da Bússola de Bordo 702
    ),
    
    periodo AS (
        SELECT
            DATE_TRUNC('month', MIN(data_venda))::date AS mes_inicio, -- arredonda para o início do mÊs
            DATE_TRUNC('month', MAX(data_venda))::date AS mes_fim
        FROM vendas_produto
    ),
    
    -- gerando a dimensão calendário
    calendario_mensal AS (
        SELECT
            generate_series(
                mes_inicio,
                mes_fim,
                INTERVAL '1 month'
            )::date AS mes
        FROM periodo
    ),
    
    vendas_mensais AS (
        SELECT
            DATE_TRUNC('month', data_venda)::date AS mes,
            SUM(quantity) AS quantidade_vendida
        FROM vendas_produto
        GROUP BY DATE_TRUNC('month', data_venda)
    )

    -- Obs: mês sem registro, entra com valor zero
    SELECT
        c.mes,
        COALESCE(v.quantidade_vendida, 0) AS quantidade_vendida
    FROM calendario_mensal c
    LEFT JOIN vendas_mensais v
        ON c.mes = v.mes
    ORDER BY c.mes;
"""

cursor.execute(codigo_sql)
dataset_vendas_mensal = cursor.fetchall()

print(f"Total de linhas: {len(dataset_vendas_mensal)}")
print('-' * 50)
for linha in dataset_vendas_mensal[:5]:
    print(linha)
print("(...)")
for linha in dataset_vendas_mensal[-5:]:
    print(linha)


Conexão realizada com sucesso!
Total de linhas: 84
--------------------------------------------------
(datetime.date(2020, 1, 1), Decimal('31.00'))
(datetime.date(2020, 2, 1), Decimal('16.00'))
(datetime.date(2020, 3, 1), Decimal('17.00'))
(datetime.date(2020, 4, 1), Decimal('32.00'))
(datetime.date(2020, 5, 1), Decimal('5.00'))
(...)
(datetime.date(2026, 8, 1), Decimal('25.00'))
(datetime.date(2026, 9, 1), Decimal('56.00'))
(datetime.date(2026, 10, 1), Decimal('29.00'))
(datetime.date(2026, 11, 1), Decimal('62.00'))
(datetime.date(2026, 12, 1), Decimal('64.00'))


In [76]:
import statistics
from sklearn.metrics import mean_absolute_error

'''
2. Modelo baseline simples, utilizando: Média móvel dos últimos 3 meses de vendas (dados anteriores à data prevista).
'''

def calcular_media_movel(vendas):
    return statistics.mean(vendas[-3:])


In [78]:
from datetime import date

'''
3. Previsão mensal de vendas para o primeiro trimestre de 2026. '''

# Separando treino e teste
treino = [
    linha for linha in dataset_vendas_mensal
    if linha[0] <= date(2025, 12, 1)
]

teste = [
    linha for linha in dataset_vendas_mensal
    if date(2026, 1, 1) <= linha[0] <= date(2026, 3, 1)
]

# Histórico inicial: somente dados até dezembro de 2025 (conjunto treino)
historico = [float(linha[1]) for linha in treino]

previsoes = []

for mes, venda_real in teste:
    # 1°: Calcula a previsão baseada nos 3 meses anteriores
    previsao = calcular_media_movel(historico)

    previsoes.append(previsao)

    # 2°: Inclui o valor real só após a previsão, para evitar data leakage (vazamento)
    historico.append(float(venda_real))


# Exibe previsões
for (mes, venda_real), previsao in zip(teste, previsoes):
    print(
        f"{mes.strftime('%Y-%m')} | "
        f"Real: {float(venda_real):.0f} | "
        f"Previsão: {previsao:.2f}"
    )


2026-01 | Real: 79 | Previsão: 38.67
2026-02 | Real: 68 | Previsão: 53.67
2026-03 | Real: 60 | Previsão: 56.33


In [80]:
'''4. Compare as previsões com os valores reais do período de teste utilizando a métrica: MAE (Mean Absolute Error)'''

valores_reais = [float(linha[1]) for linha in teste]

mae = mean_absolute_error(valores_reais, previsoes)

print("-- MAE --")
print(f"O baseline errou em média {mae:.2f} unidades por mês")


-- MAE --
O baseline errou em média 19.44 unidades por mês


## 5. Responda objetivamente:

**a. O baseline é adequado para esse produto?**

O baseline parece ter dificuldade para acompanhar a elevação da demanda, principalmente em janeiro. Como apresentou MAE de 19 unidades, NÃO é adequado esse baseline.

**b. Cite uma limitação desse método.**

A média móvel reage lentamente a mudanças na demanda, e não considera tendências ou sazonalidade em alguns meses.


## Questão 6.2 - Validação

Utilizando seu modelo treinado, qual é a soma total da previsão de vendas (arredondada para número inteiro) para o 'Bússola de Bordo 702' durante o primeiro trimestre de 2026?

38,67 + 53,67 + 56,33 = 148,67 = 149 unidades

## Questão 6.3 - Explique:

**1.	Como o baseline foi construído?**

Usei a função mean do módulo statistics para calcular a média móvel dos últimos 3 meses de vendas, utilizando os 84 registros gerados em SQL diretamente das tabelas no SGBD. 

**2.	Como evitou data leakage?**

Foram utilizados somente os valores disponíveis antes do mês que estava sendo previsto. O valor real de cada mês só foi acrescentado ao histórico (historico.append) depois que sua previsão já foi realizada.

**3.	Uma limitação do modelo proposto.**

Uma limitação é que a média móvel considera apenas os três meses anteriores, podendo demorar para acompanhar mudanças na demanda, além de não considerar tendências ou sazonalidade.


----
# Questão 7 - Sistema de recomendação

Cenário

A Marina percebeu que clientes que compram lanchas quase sempre esquecem de levar a defensa (proteção lateral). Ela quer implementar uma vitrine de "Quem comprou isso, também levou..." no site. 

Como não temos ferramentas de Big Data caras, você precisará criar um motor de recomendação, baseado na similaridade de compra dos clientes. 

Identificar qual produto deve ser recomendado junto ao item **“Motor de Popa 1949”**, com base na similaridade de comportamento de compra dos clientes.

## Tarefa:

1. Crie uma matriz de interação Usuário × Produto obedecendo às regras abaixo:

     a. Linhas: id_cliente
   
     b. Colunas: id_produto
   
     c. Valor da célula:
   
     d. 1 se o cliente comprou ao menos uma vez o produto (**usar DISTINCT**)
   
     e. 0 caso contrário
   
     f. Ignore a quantidade comprada (presença/ausência apenas)
   
2. Cálculo de Similaridade entre Produtos

     a. Calcule a Similaridade de Cosseno (Cosine Similarity) entre os vetores dos produtos
   
     b. A similaridade deve ser calculada produto × produto, com base nos clientes que compraram cada item

3. Ranking de Produtos Similares

     a. Considere o produto “Motor de Popa 1949” como item de referência

     b. Gere um ranking com os nomes dos 5 produtos mais similares a ele

     c. Desconsidere o próprio motor no ranking


## Questão 7.1 - Código em python

Script em python que:

•	Constrói a matriz usuário–item

•	Calcula a similaridade de cosseno

•	Gera o ranking de similaridade

•	Bibliotecas permitidas:
     - pandas
     - numpy
     - sklearn (opcional, para cosine similarity)


In [97]:
# 1. Criação dos datasets cliente_produto e nome_produto
# ---------------------

import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    database="nautical",
    user="postgres",
    password="postgres"
)

cursor = conexao.cursor()

print("Conexão realizada com sucesso!")

# dataset cliente vs. produto
codigo_sql = """
    SELECT DISTINCT
        o.customer_id,
        pv.product_id
    FROM orders o
    JOIN order_items oi
        ON o.id = oi.order_id
    JOIN product_variants pv
        ON oi.product_variant_id = pv.id
    ORDER BY
        o.customer_id,
        pv.product_id;
"""

cursor.execute(codigo_sql)
dataset_cliente_produto = cursor.fetchall()

# dataseto nome_produto
codigo_sql = """
    SELECT id, name
    FROM products
"""

cursor.execute(codigo_sql)
nome_produtos = cursor.fetchall()

print(f"Total de linhas cliente_produto: {len(dataset_cliente_produto)}")
print(f"Total de linhas nome_produtos: {len(nome_produtos)}")



Conexão realizada com sucesso!
Total de linhas cliente_produto: 135508
Total de linhas nome_produtos: 500


In [109]:
# 2. Cálculo de Similaridade entre Produtos
#------------------------------------------

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df = pd.DataFrame(
    dataset_cliente_produto,
    columns=["customer_id", "product_id"]
)

# Matriz cliente x produto
matriz = pd.crosstab(
    df["customer_id"],
    df["product_id"]
)

#print(matriz)


# Similaridade de cosseno transforma essa sobreposição em um número entre 0 e 1.
# Valores próximos de 1 indicam que a base de clientes que comprou os dois produtos é quase idêntica.
# Transpor matriz para produto x cliente e calcular coseno entre elas (produto vs. produto)
matriz_similaridade = cosine_similarity(matriz.T) 



# Transformar em uma tabela pandas
similaridade_df = pd.DataFrame(
    matriz_similaridade,
    index=matriz.columns,
    columns=matriz.columns
)

print(similaridade_df)


product_id       1         2         3         4         5         6    \
product_id                                                               
1           1.000000  0.226680  0.161690  0.145999  0.235263  0.194791   
2           0.226680  1.000000  0.173133  0.189424  0.231390  0.203046   
3           0.161690  0.173133  1.000000  0.120760  0.172844  0.173573   
4           0.145999  0.189424  0.120760  1.000000  0.169926  0.151925   
5           0.235263  0.231390  0.172844  0.169926  1.000000  0.188160   
...              ...       ...       ...       ...       ...       ...   
496         0.160171  0.189423  0.157810  0.169782  0.183350  0.169134   
497         0.173645  0.202057  0.213454  0.181804  0.189613  0.194878   
498         0.229478  0.183988  0.183225  0.164810  0.221060  0.205239   
499         0.104979  0.128031  0.084095  0.085152  0.147357  0.111242   
500         0.150926  0.147254  0.145792  0.156852  0.183813  0.141477   

product_id       7         8         

In [111]:
# 3. Ranking de Produtos Similares
#---------------------------------

# Tuplas extraídas do SQL para DataFrame
produtos = pd.DataFrame(
    nome_produtos,
    columns=["id", "name"]
)

# Encontrar o "Motor de Popa 1949"
produto_referencia = (
    produtos.loc[
    produtos["name"] == "Motor de Popa 1949",
    "id"
    ]
    .iloc[0]
)

#print(produto_referencia)


# Ranking dos 5 similares
ranking = (
    similaridade_df[produto_referencia]
    .drop(produto_referencia)             # eliminando os produtos que são ele mesmo, cod. 180 (diagonal com valor 1)
    .sort_values(ascending=False)         # colocando em ordem descendente
    .head(5)
)

ranking = ranking.reset_index()
ranking.columns = ["product_id", "similaridade"]

ranking = ranking.merge(
    produtos[["id", "name"]],
    left_on="product_id",
    right_on="id"
)

ranking = ranking[["product_id", "name", "similaridade"]]

print(ranking)

   product_id                name  similaridade
0         389  Motor de Popa 5331      0.256553
1         295   Cabo Náutico 2105      0.256239
2          75    Vela Mestra 1913      0.255785
3         337   Cabo Náutico 9048      0.239332
4          55    GPS Plotter 6249      0.237744


## Questão 7.2 - Validação
Qual é o nome do produto com MAIOR similaridade ao “Motor de Popa 1949”?

O produto com maior similaridade ao "Motor de Popa 1949" é o **"Motor de Popa 5331"**, com similaridade de cosseno de 0,256553.


## Questão 7.3 - Explique:
**1.	Como a matriz foi construída?**

A matriz foi construída usando os customer_id nas linhas e os product_id nas colunas. A célula recebe 1 quando o cliente comprou o produto pelo menos uma vez e 0 quando não comprou.

**2.	O que significa a similaridade de cosseno nesse contexto?**

A similaridade de cosseno significa que os vetores de clientes que compraram os dois produtos apresentam um determinado grau de proximidade.

**3.	Uma limitação desse método de recomendação.**

O modelo olha apenas para quem comprou os produtos. Produtos com poucos compradores podem gerar uma similaridade pouco confiável.

Além disso, o método não considera outras características do produto, como preço, categoria ou características técnicas.

Portanto, similaridade não significa necessariamente uma boa recomendação. 